In [1]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True }
        }
    },
    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "water": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
        }
    },
    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },
    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<45} | {'Provincia':<30}  |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} | {'Wasser':<7}"
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<45} |  {Provincia:<30}  | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7} | {water_str}")

Ciudad                                        | Provincia                       |  High   | Low    | Spanne  | Regen   | Wasser 
--------------------------------------------------------------------------------------------------------------------------------
Sevilla                                       |  Sevilla                         |   25.1 |   12.9 |      29 |    34.2 |   ---  
Lora del Río                                  |  Sevilla                         |   25.0 |   12.3 |      31 |    34.2 |   ---  
Lebrija                                       |  Sevilla                         |   24.1 |   12.1 |      28 |    35.9 |   ---  
Arrecife                                      |  Las Palmas                      |   23.9 |   18.1 |      13 |     6.4 |   ---  
Arcos de la Frontera                          |  Cádiz                           |   23.8 |   11.9 |      28 |    36.5 |   ---  
Espera                                        |  Cádiz                           |   23.8 |   11.